# Week 11 — Evaluate empirical-distribution diffusion (11d)

Eval-only counterpart to `11c_train_empirical.ipynb`, and a **cell-for-cell
mirror of `11b_evaluate.ipynb`**: same tasks, same scoreboard columns, same
plots, same physical checks — every model swapped for its empirical-target
(logit/softmax) fit. Anything you can read off 11b you can read off this
notebook at the same place, and the two scoreboards can be concatenated
because `hard_nll_direct` reuses the classical `mu_0(A)` hard gate, so the
included-block set is identical.

## Prerequisites

1. `diffusion_windows_v2.parquet` in the Week-10 artifact directory.
2. At least one `ckpt_emp_E*.ckpt` checkpoint in the same directory
   (produced by `11c_train_empirical.ipynb`).

## What differs from 11b, and why nothing downstream changes

The sampler decodes through a softmax, so every generated density is
non-negative and integrates to 1 **by construction**. The four NLL variants
11b needs (raw / normalized / raw+∫-projected / normalized+∫-projected)
therefore all collapse onto the same number here. They are still computed
and still plotted: three identical bars *is* the result — it is the direct
demonstration that the classifier-free-guidance mass-inflation leak 11b had
to close cannot open on this target.

Likewise there is no `+ p_classical` decomposition at score time (the model
predicts the full distribution), and the oracle maps cond to a per-bin
Gaussian over standardized **logits** rather than standardized residuals —
so its NLL is an upper bound on the same target the diffusion predicts.

> The **test split is reserved for the PI**. Every data-loading cell in
> this notebook filters to `split in {"train", "val"}`. Do not change that.

In [ ]:
# Standard Week 10 setup: locate the repo, install if missing.

import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, glob, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.stats import norm as sp_norm
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Bootstrap sys.path and locate artifacts.
import os, sys

# Several week directories ship a conditioned_infrastructure.py, but only the
# Week-10/11 copy defines find_week10_artifacts and the Extended* API — the
# Week-09 copy is an older stub. Force week_10 (the copy find_week10_artifacts
# itself resolves to, so no module eviction happens) to the FRONT of sys.path
# so the Week-09 stub can never shadow it, and add week_11 so eval mode can
# import its evaluation.py / empirical_infrastructure.py. Also drop any stale
# module a prior failed import may have cached as the Week-09 stub.
_week09_dir = os.path.abspath(os.path.join(repo_path, "weeks", "week_09"))
_week10_dir = os.path.abspath(os.path.join(repo_path, "weeks", "week_10"))
_week11_dir = os.path.abspath(os.path.join(repo_path, "weeks", "week_11"))
for _p in (_week09_dir, _week11_dir, _week10_dir):   # week_10 inserted last -> resolves first; week_09 trails so its conditioned_infrastructure stub never shadows but unconditioned_infrastructure stays importable
    if _p in sys.path:
        sys.path.remove(_p)
    sys.path.insert(0, _p)
sys.modules.pop("conditioned_infrastructure", None)

from conditioned_infrastructure import find_week10_artifacts
# parquet_v2 is built by 11_00_build_v2 / Part A of the training notebook; the
# raw CSV is needed by the eval phase (Task 67 Part 1). Both are listed as
# required so a missing raw CSV fails loudly at setup time.
paths = find_week10_artifacts(extra_required=[
    "data/composite_sunspot_groups_peak_area.csv",
])
# find_week10_artifacts resolves conditioned_infrastructure to the week_10
# copy (its home, where the parquet + checkpoints live) and evicts any other
# cached copy. Repoint the *module* to the week_11 copy — that's where THIS
# notebook's API lives — while keeping the week_10 artifact paths in `paths`.
if _week11_dir in sys.path:
    sys.path.remove(_week11_dir)
sys.path.insert(0, _week11_dir)
sys.modules.pop("conditioned_infrastructure", None)
sys.modules.pop("empirical_infrastructure", None)

from unconditioned_infrastructure import make_cosine_schedule
from conditioned_infrastructure import (
    block_cond_concat,
    k_run_combined,
    build_experiment_registry,
    resolve_experiment_stems,
)
from empirical_infrastructure import (
    EmpiricalDistributionDataset,
    ExtendedConditionalEmpiricalDiffusionLightning,
    load_trained_empirical_experiment,
    sample_empirical_extended,
    discover_emp_experiment_checkpoints,
    hard_nll_direct,
    EMP_CKPT_PREFIX,
)
from butterflAI_model import ButterflAIModel

import empirical_infrastructure as _ei
print(f"using empirical_infrastructure from: {_ei.__file__}")

_WEEK10_DIR = paths["week10_dir"]
PARQUET_V2  = os.path.join(_WEEK10_DIR, "diffusion_windows_v2.parquet")
CKPT_DIR    = _WEEK10_DIR

classical   = ButterflAIModel(paths["classical_weights"])

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

T = 200
alpha_np, sigma_np, _ = make_cosine_schedule(T=T, s=0.008)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the v1 parquet — we never modify it; it is here only so the split /
# cycle bookkeeping printed below matches 11b's.
# Test split is reserved for the PI.
windows_v1 = pd.read_parquet(paths["parquet_v1"])
windows_v1 = windows_v1.loc[windows_v1["split"].isin(["train", "val"])].reset_index(drop=True)
print(f"v1 parquet (train+val only): {len(windows_v1)} rows")
print(f"  splits: {windows_v1['split'].value_counts().sort_index().to_dict()}")
print(f"  cycles: {sorted(windows_v1['cycle'].unique())}")
print(f"v2 parquet target: {PARQUET_V2}")
print(f"device           : {device}")

---
## Experiment registry

The `_SPECS_EMP` list below must **mirror** the one in
`11c_train_empirical.ipynb`. The scoring loop looks up each discovered
`ckpt_emp_*.ckpt` here to recover its `consumed_keys`, architecture, and the
KL / smoothing knobs the dataset needs to be rebuilt identically. If you add a
new experiment in the training notebook, copy its spec here too — otherwise
the scoring loop will skip the checkpoint with an "unknown experiment"
warning.

Run names are **generated from the specs**, so mirroring the specs is enough.
Checkpoints trained before the rename keep their `E<n>metrics` file names on
disk and are resolved by `resolve_experiment_stems()`.


In [ ]:
# Experiment specs. MUST MIRROR _SPECS_EMP in 11c_train_empirical.ipynb.
# Because names are GENERATED from the specs, mirroring the spec list is
# enough — the notebooks cannot drift into disagreeing names for a config.
#
# Run names are GENERATED from the specs by canonical_experiment_name(), not
# typed by hand, so a name can never disagree with the config it labels:
#
#     <cond-set>_<arch>[_four][_guid][_h###][_L#]
#
#   cond-set : "base" plus added groups in the fixed order hemi < opp < traj,
#              joined by "+"  ("trajv" = traj group + its validity mask)
#   arch     : "cat" (concat) or "film" — ALWAYS stated
#   four     : fourier=True          guid : cond_dropout_p > 0
#   h###/L#  : appended only when capacity differs from _BASE_TEMPLATE
#
# Checkpoints land at ckpt_emp_<name>.ckpt (EMP_CKPT_PREFIX), so these share
# the residual family's names without colliding with its files.

_BASE_TEMPLATE = {
    "arch":           "concat",
    "consumed_keys":  ["cond_base"],
    "groups":         ["base"],
    "hidden_dim":     128,
    "n_layers":       3,
    "fourier":        False,
    "cond_dropout_p": 0.0,
    "max_epochs":     5000,
    "lr":             1e-3,
    "batch_size":     64,
    # Empirical-target specific knobs.
    "lambda_kl":      0.1,
    "alpha_smooth":   1.0,
    "n_pseudo_obs":   30.0,
}

def _spec(**overrides):
    d = dict(_BASE_TEMPLATE); d.update(overrides); return d

_SPECS_EMP = [
    # ── Base conditioning (4-D cond) × mechanism ────────────────────────────
    # The full 2×2×2 over arch, Fourier lifting and guidance, holding the
    # information content fixed: how much comes from mechanism alone?
    _spec(),                                                   # base_cat  (baseline)
    _spec(arch="film"),                                        # base_film
    _spec(fourier=True),                                       # base_cat_four
    _spec(arch="film", fourier=True),                          # base_film_four
    _spec(cond_dropout_p=0.1),                                 # base_cat_guid
    _spec(arch="film", cond_dropout_p=0.1),                    # base_film_guid
    _spec(fourier=True, cond_dropout_p=0.1),                   # base_cat_four_guid
    _spec(arch="film", fourier=True, cond_dropout_p=0.1),      # base_film_four_guid

    # ── Level 1 — one new cond group at a time, concat arch ─────────────────
    _spec(consumed_keys=["cond_base", "cond_cyclehemi"],       # base+hemi_cat
          groups=["base", "cyclehemi"]),
    _spec(consumed_keys=["cond_base", "cond_opp"],             # base+opp_cat
          groups=["base", "opp"]),
    _spec(consumed_keys=["cond_base", "cond_traj"],            # base+traj_cat
          groups=["base", "traj"]),

    # ── Levels 3–5 — best L1 cond group (opp) × mechanism ───────────────────
    _spec(arch="film",                                         # base+opp_film
          consumed_keys=["cond_base", "cond_opp"],
          groups=["base", "opp"]),
    _spec(arch="film",                                         # base+opp_film_guid
          consumed_keys=["cond_base", "cond_opp"],
          groups=["base", "opp"],
          cond_dropout_p=0.1),
    _spec(arch="film",                                         # base+opp_film_four
          consumed_keys=["cond_base", "cond_opp"],
          groups=["base", "opp"],
          fourier=True),
    _spec(arch="film",                                         # base+opp_film_four_guid
          consumed_keys=["cond_base", "cond_opp"],
          groups=["base", "opp"],
          fourier=True,
          cond_dropout_p=0.1),

    # ── Trajectory conditioning, and the opp × traj pair ────────────────────
    _spec(arch="film",                                         # base+traj_film_four
          consumed_keys=["cond_base", "cond_traj"],
          groups=["base", "traj"],
          fourier=True),
    _spec(arch="film",                                         # base+opp+traj_film_four
          consumed_keys=["cond_base", "cond_opp", "cond_traj"],
          groups=["base", "opp", "traj"],
          fourier=True),
    # Same as base+traj_film_four plus the window-validity mask — the "v".
    _spec(arch="film",                                         # base+trajv_film_four
          consumed_keys=["cond_base", "cond_traj", "cond_traj_valid"],
          groups=["base", "traj"],
          fourier=True),
]

# Keys the registry by canonical name; raises if two specs collide, which
# catches both duplicates and knobs the naming scheme doesn't yet encode.
EXPERIMENTS_EMP = build_experiment_registry(_SPECS_EMP, _BASE_TEMPLATE)

for name, cfg in EXPERIMENTS_EMP.items():
    print(f"{name:26s}: arch={cfg['arch']:6s}  consumed={cfg['consumed_keys']}  "
          f"fourier={cfg['fourier']}  cond_dropout_p={cfg['cond_dropout_p']}")


<a id="eval-mode"></a>

---
# NLL ablations across all experiments (Tasks 67–69, 71–72)

This notebook scores every checkpoint produced by the empirical training half
(`ckpt_emp_E*.ckpt` in the Week-10 artifact directory) against the same
**hard-gated NLL** metric used in 10c and 11b, and adds the same critical
diagnostic: an **oracle MLP** that maps each experiment's cond vector directly
to per-bin Gaussian parameters over the target. The oracle's NLL is an
*upper bound* on what any model can extract from a given cond set:

- Diffusion ≪ oracle  →  architecture is the bottleneck (try FiLM, CFG, Fourier).
- Oracle ≈ classical  →  the cond set itself doesn't help; try a different
  cond group or stop running that variant.

On this notebook's target the oracle predicts standardized **logits** rather
than standardized residuals, so it bounds exactly what the diffusion is being
asked to predict.

This is what makes the ablation scientifically honest: without an
oracle, a flat NLL across experiments could mean *either* "more cond
doesn't help" *or* "the architecture can't extract the new cond's
information" — two completely different fixes.

---
## Task 67 — Build per-window evaluation blocks

Re-window the raw CSV the same way 10c does (per-window, 6-monthly,
≥ 20 obs per window) and tag each window with its v2 parquet row's
*entire* cond superset — every group, normalized later per-experiment
using the corresponding checkpoint's `cond_<g>_means` / `cond_<g>_stds`
buffers. This is the **same builder with the same arguments** as 11b, so
the included-block set matches exactly and the empirical-target NLL is
directly comparable to 11b's residual-target NLL and to the classical NLL.

In [ ]:
# Task 67 Part 1 — per-window blocks tagged with their v2 cond superset.
from evaluation import build_eval_hemicycles

# Load v2 parquet (built by 11_00_build_v2 / Task 63 of the training notebook).
windows_v2 = pd.read_parquet(PARQUET_V2)
windows_v2 = windows_v2.loc[windows_v2["split"].isin(["train", "val"])].reset_index(drop=True)
# load_trained_empirical_experiment / score_checkpoint expect a `windows_aug`
# DataFrame — in the training notebook that's the in-memory build that gets
# written to the v2 parquet; here we read the same DataFrame back, so alias it.
windows_aug = windows_v2

hemicycles, GROUP_COLS = build_eval_hemicycles(
    raw_csv_path=paths["raw_csv"],
    windows_v2=windows_v2,
    classical=classical,
    splits=("train", "val"),
)

assert len(hemicycles) > 0, "rebuild the per-window blocks before continuing"
assert all("groups_raw" in blk for hc in hemicycles for blk in hc["blocks"]), \
    "every block needs a groups_raw dict"
print(f"per-window blocks built: {sum(len(hc['blocks']) for hc in hemicycles)}")
print(f"hemicycles included    : {len(hemicycles)}")
print(f"GROUP_COLS             : {list(GROUP_COLS.keys())}")

---
## Task 67 (cont) — NLL primitives, same gate as 10c / 11b

`hard_nll_direct` scores the generated density at the observed latitudes
under the *identical* `mu > mu_0(A)` hard gate that `hard_nll_combined` uses
in 11b — it just skips the `+ p_classical` step, because here the model
already predicts the whole distribution.

The four metric variants 11b needs are all defined here too, so the two
scoreboards carry the same columns:

| variant | 11b (residual target) | 11d (empirical target) |
|---|---|---|
| raw | score `p_cl + r` un-normalized | score `p` |
| normalized | divide by total mass (closes the CFG leak) | divide by total mass |
| ∫-projected | demean `r` so `∫r = 0` | clip `p ≥ 0`, renormalize to `∫p = 1` |

On the residual side those are four genuinely different numbers, because
`p_cl + r` is only a density by accident. Here the softmax decode enforces
non-negativity and unit mass by construction, so all four coincide to
float32 precision. That coincidence is the headline structural result of the
empirical framing, not an artifact — see Task 71.

In [ ]:
# Task 67 Part 2 — hard NLL primitives plus the metric-integrity and
# physical-plausibility diagnostics (evaluation.py is shared with 11b; the
# density transforms live in empirical_infrastructure.py).
from evaluation import (
    hard_nll_classical,
    sample_diagnostics,
    assemble_butterfly,
    butterfly_physical_checks,
    hemispheric_symmetry,
)
from empirical_infrastructure import (
    normalize_densities,
    project_densities_simplex,
    assemble_butterfly_direct,
)

# Bind the bin geometry once so every variant has the
# (model, hcs, densities_by_block) signature that k_run_combined expects and
# can be dropped in wherever the others are used.
def nll_raw(model, hcs, dens_by_block):
    return hard_nll_direct(model, hcs, dens_by_block, bin_width=BIN_WIDTH)

def nll_norm(model, hcs, dens_by_block):
    return hard_nll_direct(
        model, hcs, normalize_densities(dens_by_block, BIN_WIDTH),
        bin_width=BIN_WIDTH,
    )

# Simplex-projected variants. Clip negative bins and renormalize to unit
# mass before scoring — the empirical-target analog of 11b's zero-integral
# projection, which demeans each residual so it integrates to zero. Both
# enforce the constraint the true target must satisfy; here the sampler has
# already satisfied it, so these must reproduce the un-projected numbers.
def nll_raw_proj(model, hcs, dens_by_block):
    return hard_nll_direct(
        model, hcs, project_densities_simplex(dens_by_block, BIN_WIDTH),
        bin_width=BIN_WIDTH,
    )

def nll_norm_proj(model, hcs, dens_by_block):
    return hard_nll_direct(
        model, hcs,
        normalize_densities(
            project_densities_simplex(dens_by_block, BIN_WIDTH), BIN_WIDTH),
        bin_width=BIN_WIDTH,
    )

val_hcs = [hc for hc in hemicycles if hc["split"] == "val"]
nll_cl_val, det_cl_val = hard_nll_classical(classical, val_hcs)
print(f"val hemicycles: {len(val_hcs)} "
      f"({sum(len(hc['blocks']) for hc in val_hcs)} blocks)")
print(f"classical hard NLL (val): {nll_cl_val:.4f}  "
      f"(coverage {det_cl_val['coverage']:.3f})")

In [ ]:
# Sanity checks on the density transforms (11b's counterpart checks
# project_residuals_zero_integral):
#   1. normalize_densities makes every block integrate to 1
#   2. project_densities_simplex additionally makes every bin non-negative
#   3. both are exact no-ops on an already-simplex density — the property the
#      whole "raw == normalized == projected" claim rests on
_rng = np.random.default_rng(0)

# (a) adversarial input: negative bins and wrong total mass.
_bad = {("c", "h", float(i)): _rng.normal(size=15).astype(np.float32)
        for i in range(8)}
_n = normalize_densities(_bad, BIN_WIDTH)
_p = project_densities_simplex(_bad, BIN_WIDTH)
for k in _bad:
    assert abs(float(_n[k].sum() * BIN_WIDTH) - 1.0) < 1e-6, \
        f"normalize_densities failed to reach unit mass for {k}"
    assert _p[k].min() >= 0.0, f"projection left a negative bin for {k}"
    assert abs(float(_p[k].sum() * BIN_WIDTH) - 1.0) < 1e-6, \
        f"projection failed to reach unit mass for {k}"

# (b) softmax-decoded input: both transforms must be no-ops.
_logits = _rng.normal(size=(8, 15)).astype(np.float32)
_soft = np.exp(_logits) / np.exp(_logits).sum(axis=1, keepdims=True) / BIN_WIDTH
_ok = {("c", "h", float(i)): _soft[i] for i in range(8)}
for k, v in _ok.items():
    assert np.allclose(normalize_densities(_ok, BIN_WIDTH)[k], v, atol=1e-6)
    assert np.allclose(project_densities_simplex(_ok, BIN_WIDTH)[k], v, atol=1e-6)
print("density transforms: unit-mass, non-negativity and simplex no-op checks pass")


def _simplex_sanity(samples_NK15, bin_width=BIN_WIDTH):
    """Assert non-negativity + integrate-to-1 on every (15,) density vector.
    Catches a corrupted decoder before it pollutes the scoreboard."""
    nonneg = samples_NK15.min()
    mass   = samples_NK15.sum(axis=-1) * bin_width
    err    = np.abs(mass - 1.0).max()
    assert nonneg >= -1e-6, f"simplex check failed: min density = {nonneg}"
    assert err     <  1e-4, f"simplex check failed: max |Σp·Δℓ - 1| = {err}"
    return {"min_density": float(nonneg), "max_mass_err": float(err)}

---
## Task 67 (cont) — Diagnostic oracle

For each experiment's cond set, fit a tiny MLP that maps the cond
vector directly to a 15-D Gaussian over the target bins
(`mean`, `log_std`). Same `OracleMLP` / `fit_oracle` as 11b, unchanged —
the only difference is what `r_clean` carries: standardized **logits** here,
standardized residuals there. The oracle's NLL is then computed by sampling K
targets from the per-block Gaussian, decoding them through the same softmax
the diffusion uses, and feeding them through `hard_nll_direct` — the same
harness used to score the diffusion.

What the oracle answers:

- **Diffusion ≪ oracle**  →  the diffusion isn't extracting the
  information that's already in the cond. Try a stronger architecture
  (FiLM, Fourier features, larger MLP).
- **Oracle ≈ classical**  →  the cond set itself doesn't carry enough
  information about the distribution's structure. Try a different cond
  group or stop adding to this one.

In [ ]:
# Task 67 Part 3 — oracle MLP (imported from evaluation.py, shared with 11b).
from evaluation import OracleMLP, gaussian_nll, fit_oracle

---
## Task 68 — Score every checkpoint

For each discovered `ckpt_emp_E*.ckpt`:

1. Build per-experiment cond tensors for every val block by
   concatenating the right groups in `consumed_keys` order, normalized
   with the **checkpoint's own** per-group buffers (so val data uses
   train-set normalization recovered from the saved model).
2. Run K = 100 conditional samples per block using
   `sample_empirical_extended`. For CFG variants (`cond_dropout_p > 0`),
   repeat the sampling at every guidance weight in `CFG_GUIDANCE_W` and keep
   them as separate rows.
3. Fit the oracle MLP on the same (cond, standardized logit) data
   and record its val NLL as the upper bound for this cond set.
4. Plug each of the K samples into `hard_nll_direct` under all four metric
   variants; report mean and σ over K.

A simplex sanity check runs on every sampled batch (and on the oracle's
decoded samples) before anything reaches the scoreboard.

In [ ]:
# Task 68 — score every discovered checkpoint.

K = 100
CFG_GUIDANCE_W = [1.0, 1.5, 2.0, 3.0]


def score_checkpoint(name, cfg):
    """Score one empirical-target experiment against classical + oracle.

    Returns a list of dicts (one per guidance weight). Column names are
    deliberately identical to 11b's so the two scoreboards concatenate:
    {experiment, guidance_w,
     nll_mean, nll_std,                       # hard NLL on the sampled density
     nll_norm_mean, nll_norm_std,             # after renormalizing to unit mass
     nll_raw_proj_mean, nll_raw_proj_std,     # after simplex projection
     nll_norm_proj_mean, nll_norm_proj_std,   # projected then renormalized
     added_mass_mean, diversity_mean,         # metric-integrity diagnostics
     floor,
     oracle_nll_mean, oracle_nll_std,
     oracle_nll_norm_mean,                    # oracle floor, normalized metric
     oracle_nll_norm_proj_mean,               # oracle floor, projected metric
     oracle_gauss, coverage,
     min_density, max_mass_err}.              # simplex sanity (empirical only)

    On the residual target those four NLL columns differ: `p_cl + r` can go
    negative and can carry arbitrary total mass, which is exactly the leak
    classifier-free guidance exploits. Here the softmax decode fixes both, so
    they must agree to float32 precision — `max_mass_err` is the receipt.

    Note the sign convention on `added_mass_mean`: it is the median total mass
    `Σ p·Δlat`, whose ideal value is **1.0** here, against 11b's ideal of 0.0
    for a residual that adds no net mass.
    """
    # 1. Load checkpoint and datasets.
    lit, train_ds, val_ds, total_dim = load_trained_empirical_experiment(
        name=name, cfg=cfg, windows_aug=windows_aug,
        classical=classical, bin_centers=BIN_CENTERS,
        ckpt_dir=CKPT_DIR, alpha_np=alpha_np, sigma_np=sigma_np,
        bin_width=BIN_WIDTH,
    )

    # 2. Build per-val-block cond tensors.
    keys, cond_tensor = block_cond_concat(val_hcs, lit, cfg, train_ds)
    N = cond_tensor.shape[0]

    # 3. Fit the oracle on the same cond set.
    #    Build (cond, r_clean) pairs from the dataset; r_clean carries
    #    standardized logits on this target.
    train_cond = torch.stack([
        torch.cat([train_ds[i][k] for k in cfg["consumed_keys"]], dim=-1)
        for i in range(len(train_ds))
    ])
    train_r = torch.stack([train_ds[i]["r_clean"] for i in range(len(train_ds))])

    val_cond = torch.stack([
        torch.cat([val_ds[i][k] for k in cfg["consumed_keys"]], dim=-1)
        for i in range(len(val_ds))
    ])
    val_r = torch.stack([val_ds[i]["r_clean"] for i in range(len(val_ds))])

    oracle_model, oracle_gauss = fit_oracle(train_cond, train_r, val_cond, val_r)

    # Oracle samples: draw K logit vectors per val block from the oracle's
    # predicted Gaussian, de-standardize, softmax to a density, and score
    # under all the metric variants the diffusion is scored under.
    oracle_model.eval()
    with torch.no_grad():
        o_mean, o_log_std = oracle_model(cond_tensor)
    o_std = torch.exp(o_log_std)
    torch.manual_seed(0)
    oracle_std_samples = (
        o_mean.unsqueeze(1)
        + o_std.unsqueeze(1) * torch.randn(N, K, 15)
    )
    oracle_logits = oracle_std_samples * train_ds.logit_stds + train_ds.logit_means
    oracle_phys   = (torch.softmax(oracle_logits, dim=-1) / BIN_WIDTH).numpy()
    _simplex_sanity(oracle_phys)
    oracle_nlls, oracle_floors = k_run_combined(
        nll_raw, classical, val_hcs, keys, oracle_phys,
    )
    oracle_nlls_n, _ = k_run_combined(
        nll_norm, classical, val_hcs, keys, oracle_phys,
    )
    oracle_nlls_np, _ = k_run_combined(
        nll_norm_proj, classical, val_hcs, keys, oracle_phys,
    )

    # Reference per-bin spread for the diversity diagnostic: the empirical
    # density's own scatter across training windows (11b uses the residual
    # dataset's bin_stds for the same purpose).
    bin_s = np.asarray(train_ds.emp_dens, dtype=float).std(axis=0)

    # 4. Sample from the diffusion model and score.
    is_cfg = cfg.get("cond_dropout_p", 0.0) > 0
    guidance_ws = CFG_GUIDANCE_W if is_cfg else [0.0]

    results = []
    for w in guidance_ws:
        cond_K = cond_tensor.repeat_interleave(K, dim=0)
        torch.manual_seed(0)
        samples_flat = sample_empirical_extended(
            lit, cond_K, guidance_w=w, bin_width=BIN_WIDTH, device=device,
        ).cpu().numpy()
        samples_NK15 = samples_flat.reshape(N, K, 15)
        sanity = _simplex_sanity(samples_NK15)

        nlls, floors = k_run_combined(
            nll_raw, classical, val_hcs, keys, samples_NK15,
        )
        nlls_n, _ = k_run_combined(
            nll_norm, classical, val_hcs, keys, samples_NK15,
        )
        nlls_rp, _ = k_run_combined(
            nll_raw_proj, classical, val_hcs, keys, samples_NK15,
        )
        nlls_np, _ = k_run_combined(
            nll_norm_proj, classical, val_hcs, keys, samples_NK15,
        )
        diag = sample_diagnostics(samples_NK15, bin_s, BIN_WIDTH)

        results.append({
            "experiment":                name,
            "guidance_w":                w,
            "nll_mean":                  nlls.mean(),
            "nll_std":                   nlls.std(),
            "nll_norm_mean":             nlls_n.mean(),
            "nll_norm_std":              nlls_n.std(),
            "nll_raw_proj_mean":         nlls_rp.mean(),
            "nll_raw_proj_std":          nlls_rp.std(),
            "nll_norm_proj_mean":        nlls_np.mean(),
            "nll_norm_proj_std":         nlls_np.std(),
            "added_mass_mean":           diag["added_mass"],
            "diversity_mean":            diag["diversity"],
            "floor":                     floors.mean(),
            "oracle_nll_mean":           oracle_nlls.mean(),
            "oracle_nll_std":            oracle_nlls.std(),
            "oracle_nll_norm_mean":      oracle_nlls_n.mean(),
            "oracle_nll_norm_proj_mean": oracle_nlls_np.mean(),
            "oracle_gauss":              oracle_gauss,
            "coverage":                  det_cl_val["coverage"],
            "min_density":               sanity["min_density"],
            "max_mass_err":              sanity["max_mass_err"],
        })

    return results

In [ ]:
# Discover trained checkpoints and score them. resolve_experiment_stems()
# matches each on-disk stem to a canonical experiment name, accepting the
# pre-rename E<n>metrics stems so nothing had to be renamed or retrained.
# Stems with no spec are skipped with a warning instead of aborting the run.
# Rows are labeled with the CANONICAL name, so no E-numbers reach the figures.
_ckpts = discover_emp_experiment_checkpoints(_WEEK10_DIR)
_known, _unknown = resolve_experiment_stems(_ckpts, EXPERIMENTS_EMP)
if _unknown:
    print(f"WARNING: skipping checkpoints with no EXPERIMENTS_EMP spec "
          f"(add a spec to _SPECS_EMP in both notebooks if you want to score "
          f"them): {_unknown}")
STEM_BY_NAME = dict(_known)     # canonical name -> on-disk stem, used downstream
print(f"scoring checkpoints: {[c for c, _ in _known]}")

all_rows = []
for _canon, _stem in _known:
    print(f"scoring {_canon} (ckpt_emp_{_stem}.ckpt) ...")
    _rows = score_checkpoint(_stem, EXPERIMENTS_EMP[_canon])
    for _r in _rows:
        _r["experiment"] = _canon      # label by canonical name, not disk stem
    all_rows.extend(_rows)

scoreboard = pd.DataFrame(all_rows)
scoreboard["classical"] = nll_cl_val
print(scoreboard.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# The structural claim of the empirical target, checked rather than asserted
# in prose: the four metric variants must agree.
if not scoreboard.empty:
    _spread = float(np.nanmax(np.abs(
        scoreboard[["nll_mean", "nll_norm_mean",
                    "nll_raw_proj_mean", "nll_norm_proj_mean"]].to_numpy()
        - scoreboard[["nll_mean"]].to_numpy()
    )))
    print(f"\nmax |variant - raw| NLL across all rows: {_spread:.2e}  "
          f"(0 to float precision = no mass leak to close)")

---
## Distributional scorecard — does NLL agree with the eye?

NLL is pointwise and tail-dominated; it can rank the smooth classical Gaussian
above a visibly more realistic model. This cell scores the same val windows
with **distributional** metrics that track shape and calibration:

- **EMD** — mean Wasserstein-1 between generated and empirical latitude density
  (degrees; lower = better).
- **energy** — multivariate set-vs-point distance over the 15-bin vector.
- **crps_mu** — calibration+sharpness of the per-window mean latitude (Sporer point).
- **mu_mae / sigma_mae** — drift and spread errors (degrees).

Read it next to the NLL scoreboard: where a model beats classical here but not on
NLL, the realism your eye sees is real and NLL is the wrong sole arbiter.

In [ ]:
# Task 73 — distributional scorecard, with a GUIDANCE SWEEP for CFG models.
# Non-CFG models have no trained null embedding, so guidance is meaningless for
# them — they are scored unguided only. CFG-trained models (cond_dropout_p>0)
# are swept over W_SWEEP so EMD/energy/CRPS can be read as a function of w,
# directly comparable to the NLL-vs-w sweep in the scoreboard above.
import scipy.stats as _sst
from evaluation import distributional_scorecard

M_ENS   = 32
W_SWEEP = [0.0, 0.5, 1.0, 1.5, 2.0]

_blk_lut = {(hc["cycle"], hc["hemisphere"], blk["center_decimal"]): (hc, blk)
            for hc in val_hcs for blk in hc["blocks"]}

def _par_emp(hc, blk):
    A = hc["amplitude"]; mu = float(classical.mu(blk["tau"]))
    sigma = float(classical.sigma(mu, A))
    par = _sst.norm.pdf(BIN_CENTERS, loc=mu, scale=sigma)
    emp = np.histogram(blk["lats"], bins=LAT_BINS, density=True)[0]
    return par, emp

def _scorecard_at(lit, cfg, train_ds, w):
    """Distributional report for one model at one guidance weight.

    Same as 11b's helper except the model density goes in directly instead of
    as `par + resid` — there is no classical pedestal on this target.
    """
    mb, eb, tb = {}, {}, {}
    for hc in val_hcs:
        keys, cond = block_cond_concat([hc], lit, cfg, train_ds)
        if not keys:
            continue
        torch.manual_seed(0)
        dens = sample_empirical_extended(
            lit, cond.repeat_interleave(M_ENS, dim=0),
            guidance_w=w, bin_width=BIN_WIDTH, device=device,
        ).cpu().numpy().reshape(len(keys), M_ENS, 15)
        for i, key in enumerate(keys):
            hit = _blk_lut.get(tuple(key))
            if hit is None:
                continue
            _, emp = _par_emp(*hit)
            mb[key] = dens[i]; eb[key] = emp; tb[key] = hit[1]["tau"]
    return None if not mb else distributional_scorecard(
        mb, eb, BIN_CENTERS, BIN_WIDTH, tb)

# Classical baseline (deterministic; guidance n/a).
cl_model, cl_emp, cl_tau = {}, {}, {}
for hc in val_hcs:
    mu0A = float(classical.mu_0(hc["amplitude"]))
    for blk in hc["blocks"]:
        if float(classical.mu(blk["tau"])) > mu0A:
            continue
        key = (hc["cycle"], hc["hemisphere"], blk["center_decimal"])
        par, emp = _par_emp(hc, blk)
        cl_model[key] = par[None, :]; cl_emp[key] = emp; cl_tau[key] = blk["tau"]

rows = []
rep = distributional_scorecard(cl_model, cl_emp, BIN_CENTERS, BIN_WIDTH, cl_tau)
rows.append({"experiment": "classical", "guidance_w": np.nan,
             **{k: rep[k] for k in ("emd","energy","crps_mu","mu_mae","sigma_mae","n_blocks")}})

# Resolved independently of the scoring cell: canonical name + disk stem.
_known18, _ = resolve_experiment_stems(
    discover_emp_experiment_checkpoints(_WEEK10_DIR), EXPERIMENTS_EMP)
for name, _stem in _known18:
    cfg = EXPERIMENTS_EMP[name]
    lit, train_ds, _, _ = load_trained_empirical_experiment(
        name=_stem, cfg=cfg, windows_aug=windows_aug, classical=classical,
        bin_centers=BIN_CENTERS, ckpt_dir=CKPT_DIR,
        alpha_np=alpha_np, sigma_np=sigma_np, bin_width=BIN_WIDTH)
    # Sweep guidance only for CFG-trained models; others scored unguided.
    weights = W_SWEEP if cfg.get("cond_dropout_p", 0.0) > 0.0 else [0.0]
    for w in weights:
        rep = _scorecard_at(lit, cfg, train_ds, w)
        if rep is None:
            continue
        rows.append({"experiment": name, "guidance_w": w,
                     **{k: rep[k] for k in ("emd","energy","crps_mu","mu_mae","sigma_mae","n_blocks")}})

scorecard_df = pd.DataFrame(rows)
print(scorecard_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# Does guidance ever help the distributional metrics? Compare each CFG model's
# best-EMD weight against its own w=0 row.
print("\nguidance sweep verdict (CFG models, by EMD):")
_swept = scorecard_df[scorecard_df["experiment"] != "classical"]
for name, grp in _swept.groupby("experiment"):
    if len(grp) <= 1:
        continue
    best = grp.loc[grp["emd"].idxmin()]
    emd0 = float(grp.loc[grp["guidance_w"] == 0.0, "emd"].iloc[0])
    verdict = ("HELPS" if best["guidance_w"] > 0 and best["emd"] < emd0 - 1e-4
               else "no help (w=0 best)")
    print(f"  {name}: w=0 EMD {emd0:.4f} | best w={best['guidance_w']:.1f} "
          f"EMD {best['emd']:.4f}  -> {verdict}")
print("\nlower EMD / energy / crps_mu / *_mae = better.")

---
## Task 69 — Headline plot

Bar chart, val split: classical baseline plus every experiment's NLL,
with K-σ error bars. Oracle NLL per experiment overlaid as a
horizontal dashed marker to make the "what's achievable from this cond
set" boundary visible.

For any CFG variant (`cond_dropout_p > 0`), the bar shown is the
best-NLL guidance setting; a secondary panel sweeps the guidance
weight `w` so you can see the guidance vs NLL trade.

Reading the result:

- `nll_mean < classical` → the empirical-target framing solved the problem
  the residual framing couldn't (primary positive result).
- `oracle_nll_mean < classical` while `nll_mean ≈ classical` → cond carries
  information about the target, but the diffusion isn't extracting it —
  architecture bottleneck; try FiLM / Fourier / longer training.
- `oracle_nll_mean ≈ classical` → switching targets alone wasn't enough for
  this cond set; add cond groups.

Panel 1 plots the raw, normalized and projected bars as 11b does. Expect
three identical heights — on this target that is the finding, not a bug.

In [ ]:
# Task 69 — headline plot + CFG sweep + per-hemicycle breakdown.
#
# Three metrics are plotted side by side, exactly as in 11b:
#   - raw hard NLL on the sampled density;
#   - renormalized (divide each block's density by its total mass);
#   - simplex-projected + renormalized.
# In 11b, where raw and normalized disagree the raw metric is lying, and
# where normalized and projected disagree the model is drifting off the
# integrate-to-zero constraint. Here all three coincide by construction —
# the panel is the visual receipt that neither failure mode is available.

# Best raw-NLL guidance_w per experiment. The same row carries that
# setting's normalized and projected NLL.
best_idx = scoreboard.groupby("experiment")["nll_mean"].idxmin()
best_rows = scoreboard.loc[best_idx].reset_index(drop=True)

has_cfg = (scoreboard["guidance_w"] > 0.0).any()
n_panels = 3 if has_cfg else 2
fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 5))

# ── Panel 1: raw vs normalized vs projected normalized val NLL ─────
ax1 = axes[0]
labels = list(best_rows["experiment"])
xr = np.arange(len(labels))
wbar = 0.27
ax1.bar(xr - wbar, best_rows["nll_mean"], wbar,
        yerr=best_rows["nll_std"], color="C2", capsize=2, label="raw NLL")
ax1.bar(xr,        best_rows["nll_norm_mean"], wbar,
        yerr=best_rows["nll_norm_std"], color="C1", capsize=2,
        label="normalized NLL")
ax1.bar(xr + wbar, best_rows["nll_norm_proj_mean"], wbar,
        yerr=best_rows["nll_norm_proj_std"], color="C4", capsize=2,
        label="normalized + simplex proj")
# Oracle floor (projected normalized) as a dashed tick spanning each
# experiment's group — same constraint as the model's projected bar, so it
# is the fair upper bound under this metric.
for i, (_, row) in enumerate(best_rows.iterrows()):
    ax1.plot([i - 1.5 * wbar, i + 1.5 * wbar],
             [row["oracle_nll_norm_proj_mean"]] * 2,
             "k--", lw=1.2, label="oracle (proj)" if i == 0 else None)
ax1.axhline(nll_cl_val, color="C0", lw=1.5, label="classical")
ax1.set_xticks(xr)
ax1.set_xticklabels(labels, rotation=35, ha="right", rotation_mode="anchor")
ax1.set_ylabel("hard NLL  (lower is better)")
ax1.set_title("Val NLL: raw vs normalized vs projected")
ax1.legend(fontsize=8)

# ── Panel 2: CFG guidance sweep, all three metrics ─────────────────
if has_cfg:
    ax2 = axes[1]
    cfg_rows = scoreboard[scoreboard["guidance_w"] > 0.0]
    for exp_name, grp in cfg_rows.groupby("experiment"):
        grp = grp.sort_values("guidance_w")
        line, = ax2.plot(grp["guidance_w"], grp["nll_mean"],
                         marker="o", label=f"{exp_name} raw")
        ax2.plot(grp["guidance_w"], grp["nll_norm_mean"],
                 marker="s", ls="--", color=line.get_color(),
                 label=f"{exp_name} norm")
        ax2.plot(grp["guidance_w"], grp["nll_norm_proj_mean"],
                 marker="^", ls=":", color=line.get_color(),
                 label=f"{exp_name} norm+proj")
    ax2.axhline(nll_cl_val, color="C0", ls=":", label="classical")
    ax2.set_xlabel("guidance weight $w$")
    ax2.set_ylabel("hard NLL")
    ax2.set_title("CFG sweep: raw and norm track each other (no leak)")
    ax2.legend(fontsize=6, ncol=2)
    panel_hc = axes[2]
else:
    panel_hc = axes[1]

# ── Panel 3: per-hemicycle breakdown (projected normalized metric) ──
# Same honest-metric selection criterion as 11b.
best_sel  = scoreboard.loc[scoreboard["nll_norm_proj_mean"].idxmin()]
best_name = best_sel["experiment"]
best_cfg  = EXPERIMENTS_EMP[best_name]
best_w    = float(best_sel["guidance_w"])

_raw_best = best_rows.loc[best_rows["nll_mean"].idxmin(), "experiment"]
if best_name != _raw_best:
    print(f"note: raw NLL favors {_raw_best}; the projected normalized "
          f"metric favors {best_name} — showcasing it. (On this target the "
          f"two metrics are identical, so any disagreement is a tie-break.)")

lit_best, train_ds_best, _, _ = load_trained_empirical_experiment(
    name=STEM_BY_NAME[best_name], cfg=best_cfg, windows_aug=windows_aug,
    classical=classical, bin_centers=BIN_CENTERS, ckpt_dir=CKPT_DIR,
    alpha_np=alpha_np, sigma_np=sigma_np, bin_width=BIN_WIDTH,
)

cycle_rows = []
for hc in val_hcs:
    nll_cl_hc, _ = hard_nll_classical(classical, [hc])
    keys_hc, cond_hc = block_cond_concat([hc], lit_best, best_cfg,
                                          train_ds_best)
    if not keys_hc:
        continue
    cond_K_hc = cond_hc.repeat_interleave(K, dim=0)
    torch.manual_seed(0)
    samp_hc = sample_empirical_extended(
        lit_best, cond_K_hc, guidance_w=best_w, bin_width=BIN_WIDTH,
        device=device,
    ).cpu().numpy().reshape(len(keys_hc), K, 15)
    nlls_hc, _ = k_run_combined(
        nll_norm_proj, classical, [hc], keys_hc, samp_hc,
    )
    cycle_rows.append({
        "label": f"{hc['cycle']:02d}{hc['hemisphere'][0]}",
        "classical": nll_cl_hc,
        "diff_mean": nlls_hc.mean(),
        "diff_std":  nlls_hc.std(),
    })
breakdown = pd.DataFrame(cycle_rows,
                         columns=["label", "classical", "diff_mean", "diff_std"])

x3 = np.arange(len(breakdown))
w3 = 0.35
panel_hc.bar(x3 - w3 / 2, breakdown["classical"], width=w3,
             color="C0", label="classical")
panel_hc.bar(x3 + w3 / 2, breakdown["diff_mean"],
             yerr=breakdown["diff_std"], width=w3,
             color="C4", label=f"{best_name} (proj)", capsize=3)
panel_hc.set_xticks(x3)
panel_hc.set_xticklabels(breakdown["label"], rotation=35, ha="right", rotation_mode="anchor")
panel_hc.set_ylabel("hard NLL (normalized + simplex proj)")
panel_hc.set_title(f"Per-hemicycle: {best_name} (w={best_w:.1f})")
panel_hc.legend(fontsize=8)

fig.suptitle("Week 11 empirical-target ablation results (val split)", fontsize=12)
fig.tight_layout()
plt.show()

---
## Task 71 — Metric integrity: the leak 11b had to close

In 11b the raw `hard_nll_combined` scores the *un-normalized* density
`p_cl + residual` at the observed latitudes. Classifier-free guidance
amplifies the residual (`eps = (1+w)·eps_cond − w·eps_null`), so as `w`
grows the model can pile extra density onto exactly the occupied bins
and drive the raw NLL arbitrarily negative — over-spent mass in the
empty bins is floored and never scored.

On the empirical target guidance is applied in **logit** space and the sample
is decoded through a softmax, so the mass is renormalized after guidance,
every bin is non-negative, and the leak is closed at the level of the
parameterization rather than by patching the metric. This panel runs the same
three diagnostics 11b runs so you can see that directly. All three use
**robust** statistics (median, IQR) because CFG can still produce a small
fraction of extreme samples that dominate mean/std:

- **total mass** = median of `Σ p·Δlat` across blocks×K. **Ideal = 1.0** here
  (11b's residual version has ideal 0, being "added" mass). A flat line at 1
  across `w` is the leak being structurally impossible.
- **diversity ratio** = median of (across-K density IQR ÷ empirical density
  IQR). Healthy = 1. Values that drift away in *either* direction are
  pathological: `<1` is mode collapse (samples too tight), `>1` is CFG
  *explosion*. Guidance can still cause both here — it just can't cheat the
  metric while doing so.
- **floor fraction** = how often the sampled density fell below `eps` at an
  observed latitude. Softmax outputs are strictly positive, so a nonzero
  value means the model put essentially no mass where spots actually emerged
  — a real modeling failure rather than a negative-density artifact.

In [ ]:
# Task 71 — metric-integrity diagnostics for the CFG experiments.
cfg_rows = scoreboard[scoreboard["guidance_w"] > 0.0]
if cfg_rows.empty:
    print("No classifier-free-guidance experiments scored — nothing to diagnose.")
else:
    panels = [
        ("added_mass_mean", "total mass  (median Σ p·Δlat)",            1.0),
        ("diversity_mean",  "diversity ratio (sample IQR / empirical)", 1.0),
        ("floor",           "floor fraction (density below eps)",       0.0),
    ]
    titles = ["Mass is conserved by construction",
              "Diversity ratio drifts from healthy (=1)",
              "Unmodeled-latitude clamps"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (col, ylabel, ref), title in zip(axes, panels, titles):
        for exp_name, grp in cfg_rows.groupby("experiment"):
            grp = grp.sort_values("guidance_w")
            ax.plot(grp["guidance_w"], grp[col], marker="o", label=exp_name)
        ax.axhline(ref, color="k", ls=":", lw=1, label="ideal")
        ax.set_xlabel("guidance weight $w$")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.legend(fontsize=8)
    # The diversity ratio can shoot far above 1 when CFG explodes a few
    # samples, which then dominates the IQR — log scale keeps both that
    # regime and the "collapse" regime (<1) legible on one panel.
    axes[1].set_yscale("symlog", linthresh=1.0)
    fig.suptitle("Task 71 — Metric integrity diagnostics (CFG experiments)",
                 fontsize=12)
    fig.tight_layout(); plt.show()

    # One-line verdict per CFG experiment at its raw-best guidance weight.
    print("At each experiment's raw-best guidance weight:")
    for exp_name, grp in cfg_rows.groupby("experiment"):
        r = grp.loc[grp["nll_mean"].idxmin()]
        print(f"  {exp_name}: w={r['guidance_w']:.1f}  "
              f"raw NLL {r['nll_mean']:+.3f} -> normalized {r['nll_norm_mean']:+.3f}  "
              f"| total_mass {r['added_mass_mean']:.3f}, "
              f"diversity {r['diversity_mean']:.2f}, floor {r['floor']:.3f}")

---
## Task 72 — Physical plausibility: assemble the butterfly diagram

NLL — even on a properly normalized simplex — only scores density at the
latitudes where spots were actually observed. It cannot tell you whether the
*generated* distribution is physically sensible. So we stack each variant's
per-window densities into a latitude-vs-time **butterfly diagram** and
check three physical signatures:

- **Spörer's law** — the emergence-latitude centroid should drift
  *equatorward* over a hemicycle (negative slope, deg/yr).
- **Latitude bounds** — emergence should sit in the Spörer zone
  (±5–40°); mass leaking outside is unphysical.
- **Hemispheric symmetry** — where both hemispheres of a cycle are in
  the val split, their drift / band should be comparable.

Unlike 11b there is no `+ p_classical` step — the generated densities go in
directly, via `assemble_butterfly_direct`. That helper applies the *same*
`mu > mu_0(A)` hard gate as `assemble_butterfly`, so the surviving window set
matches 11b's panels column for column and the two figures can be compared
side by side.

In [ ]:
# Task 72 — assemble per-window densities into butterfly diagrams for every
# experiment, alongside empirical + classical baselines. The classical hard
# gate is w-independent, so the set of surviving windows matches across all
# variants and the panels stay aligned.
if scoreboard.empty:
    print("no scored experiments — nothing to plot")
else:
    # Best (lowest projected-normalized NLL) row per experiment — same
    # honest-metric selection used for the per-hemicycle panel in Task 69.
    best_idx_by_exp = scoreboard.groupby("experiment")["nll_norm_proj_mean"].idxmin()
    best_by_exp = scoreboard.loc[best_idx_by_exp].set_index("experiment")

    # Showcase hemicycle: the val one with the most windows.
    hc_show  = max(val_hcs, key=lambda hc: len(hc["blocks"]))
    tag_show = f"{hc_show['cycle']:02d}{hc_show['hemisphere'][0]}"

    def _mean_density_for(hc, lit_m, train_ds_m, cfg_m, w):
        """Mean-over-K generated density per block: key -> (15,)."""
        keys_hc, cond_hc = block_cond_concat([hc], lit_m, cfg_m, train_ds_m)
        if not keys_hc:
            return {}
        cond_K_hc = cond_hc.repeat_interleave(K, dim=0)
        torch.manual_seed(0)
        samp = sample_empirical_extended(
            lit_m, cond_K_hc, guidance_w=w, bin_width=BIN_WIDTH, device=device,
        ).cpu().numpy().reshape(len(keys_hc), K, 15)
        return {key: samp[i].mean(axis=0) for i, key in enumerate(keys_hc)}

    # Generate one butterfly per experiment.
    gen_by_exp = {}
    t_ref = None
    for exp_name in best_by_exp.index:
        row = best_by_exp.loc[exp_name]
        cfg_m = EXPERIMENTS_EMP[exp_name]
        w_m = float(row["guidance_w"])
        lit_m, train_ds_m, _, _ = load_trained_empirical_experiment(
            name=STEM_BY_NAME[exp_name], cfg=cfg_m, windows_aug=windows_aug,
            classical=classical, bin_centers=BIN_CENTERS, ckpt_dir=CKPT_DIR,
            alpha_np=alpha_np, sigma_np=sigma_np, bin_width=BIN_WIDTH,
        )
        mean_dens = _mean_density_for(hc_show, lit_m, train_ds_m, cfg_m, w_m)
        m, t = assemble_butterfly_direct(classical, hc_show, mean_dens, BIN_CENTERS)
        gen_by_exp[exp_name] = (m, t, w_m, float(row["nll_norm_proj_mean"]))
        if t_ref is None or t.size > t_ref.size:
            t_ref = t

    # Empirical density at the shared (gated) windows.
    _blk_by_t = {float(b["center_decimal"]): b for b in hc_show["blocks"]}
    emp_show = (np.array([np.histogram(_blk_by_t[t]["lats"], bins=LAT_BINS,
                                       density=True)[0] for t in t_ref])
                if t_ref is not None and t_ref.size else np.empty((0, 15)))

    # Classical-only butterfly: assemble with zero residuals so we reuse the
    # exact same gating + ordering as the model panels.
    zero_resid = {(hc_show["cycle"], hc_show["hemisphere"], float(t)): np.zeros(15)
                  for t in (t_ref if t_ref is not None else [])}
    cl_show, _ = assemble_butterfly(classical, hc_show, zero_resid, BIN_CENTERS)

    if t_ref is None or t_ref.size == 0:
        print(f"hemicycle {tag_show}: no windows survive the classical gate.")
    else:
        panels = [("empirical", emp_show, t_ref, None, None),
                  ("classical", cl_show, t_ref, None, nll_cl_val)]
        panels += [(name, m, t, w, nll)
                   for name, (m, t, w, nll) in gen_by_exp.items()]

        # vmax = max((m.max() for _, m, _, _, _ in panels if m.size), default=1.0)
        vmax = 0.15  # shared color scale across panels — hardcoded for legibility

        n_panels = len(panels)
        ncols = min(3, n_panels)
        nrows = int(np.ceil(n_panels / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows),
                                 sharey=True, squeeze=False)
        axes_flat = axes.flatten()

        for ax, (name, m, t, w, nll) in zip(axes_flat, panels):
            if w is None and nll is None:
                title = name
            elif w is None:
                title = f"{name}  NLL={nll:.3f}"
            else:
                title = f"{name}  w={w:.1f}  NLL={nll:.3f}"
            im = ax.imshow(m.T, origin="lower", aspect="auto", vmin=0, vmax=vmax,
                           extent=[t.min(), t.max(),
                                   LAT_BINS[0], LAT_BINS[-1]], cmap="magma")
            ax.axhline(5, color="c", ls=":", lw=1)
            ax.axhline(40, color="c", ls=":", lw=1)
            ax.set_xlabel("year")
            ax.set_title(title, fontsize=10)
        for ax in axes_flat[len(panels):]:
            ax.axis("off")
        for r in range(nrows):
            axes[r, 0].set_ylabel("|latitude| (°)")

        fig.colorbar(im, ax=axes_flat[:len(panels)].tolist(),
                     fraction=0.025, pad=0.02, label="density")
        fig.suptitle(f"Butterfly diagram: {tag_show}  "
                     f"(Spörer band 5–40° dotted)",
                     fontsize=12)
        plt.show()

        print(f"\nPhysical checks for {tag_show}:")
        c = butterfly_physical_checks(emp_show, t_ref, BIN_CENTERS)
        print(f"  {'empirical':>14}: "
              f"sporer_slope={c['sporer_slope']:+.3f} deg/yr "
              f"(equatorward={c['sporer_ok']}), "
              f"in_band={c['in_band_fraction']:.3f}, "
              f"centroid {c['centroid_start']:.1f}->{c['centroid_end']:.1f}")
        c = butterfly_physical_checks(cl_show, t_ref, BIN_CENTERS)
        print(f"  {'classical':>14}: "
              f"sporer_slope={c['sporer_slope']:+.3f} deg/yr "
              f"(equatorward={c['sporer_ok']}), "
              f"in_band={c['in_band_fraction']:.3f}, "
              f"centroid {c['centroid_start']:.1f}->{c['centroid_end']:.1f}, "
              f"NLL={nll_cl_val:.3f}")
        for name, (m, t, w, nll) in gen_by_exp.items():
            c = butterfly_physical_checks(m, t, BIN_CENTERS)
            tag = f"{name} w={w:.1f}"
            print(f"  {tag:>14}: "
                  f"sporer_slope={c['sporer_slope']:+.3f} deg/yr "
                  f"(equatorward={c['sporer_ok']}), "
                  f"in_band={c['in_band_fraction']:.3f}, "
                  f"centroid {c['centroid_start']:.1f}->{c['centroid_end']:.1f}, "
                  f"NLL={nll:.3f}")

    # Hemispheric symmetry across all val hemicycles, evaluated at each
    # experiment's best w. Same selection criterion as the per-experiment
    # panels above (lowest projected-normalized NLL).
    sym_by_exp = {}
    for exp_name in best_by_exp.index:
        row = best_by_exp.loc[exp_name]
        cfg_m = EXPERIMENTS_EMP[exp_name]
        w_m = float(row["guidance_w"])
        lit_m, train_ds_m, _, _ = load_trained_empirical_experiment(
            name=STEM_BY_NAME[exp_name], cfg=cfg_m, windows_aug=windows_aug,
            classical=classical, bin_centers=BIN_CENTERS, ckpt_dir=CKPT_DIR,
            alpha_np=alpha_np, sigma_np=sigma_np, bin_width=BIN_WIDTH,
        )
        checks_by_hc = {}
        for hc in val_hcs:
            gm, gt = assemble_butterfly_direct(
                classical, hc,
                _mean_density_for(hc, lit_m, train_ds_m, cfg_m, w_m),
                BIN_CENTERS,
            )
            if gm.shape[0] >= 2:
                checks_by_hc[(hc["cycle"], hc["hemisphere"])] = \
                    butterfly_physical_checks(gm, gt, BIN_CENTERS)
        sym_by_exp[exp_name] = hemispheric_symmetry(checks_by_hc)

    any_sym = any(sym for sym in sym_by_exp.values())
    if any_sym:
        print("\nHemispheric symmetry (cycles with both N and S in val):")
        for exp_name, sym in sym_by_exp.items():
            w_m = float(best_by_exp.loc[exp_name, "guidance_w"])
            if not sym:
                continue
            print(f"  {exp_name} (w={w_m:.1f}):")
            for cyc, d in sym.items():
                print(f"    cycle {cyc}: "
                      f"|slope_N - slope_S|={d['slope_diff']:.3f}, "
                      f"|in_band_N - in_band_S|={d['in_band_diff']:.3f}")
    else:
        print("\nHemispheric symmetry: no cycle has both N and S in the val "
              "split (val is hemisphere-mixed) — per-hemicycle checks only.")

---
## Handoff back to the PI

The headline numbers in `scoreboard` answer two questions:

1. **Does any variant beat the classical baseline on val?** If yes, the
   empirical-target diffusion has earned its place in the final pipeline.
2. **Where is the bottleneck — information or architecture?** Compare
   each row's `nll_mean` to its `oracle_nll_mean`. A large gap means
   the cond set has more information than the diffusion is extracting
   (architecture-bound). A small gap with the oracle near classical
   means the cond set isn't carrying enough information — that line of
   experiments is exhausted; try a different cond group.

And one this notebook answers that 11b could not:

3. **Did the target framing, rather than the architecture, cause the
   residual model's ceiling?** Read this `scoreboard` next to 11b's — the
   columns are identical and the gated block set is identical, so the two
   are directly subtractable per experiment.

The PI will run the test set evaluation on whatever variant the val
results recommend.

In [ ]:
# Task 72 (figure variant) — same assembly as above with paper-ready panel
# titles and no per-panel metric annotations. Kept separate so the diagnostic
# version above stays fully labeled.
if scoreboard.empty:
    print("no scored experiments — nothing to plot")
else:
    best_idx_by_exp = scoreboard.groupby("experiment")["nll_norm_proj_mean"].idxmin()
    best_by_exp = scoreboard.loc[best_idx_by_exp].set_index("experiment")

    hc_show  = max(val_hcs, key=lambda hc: len(hc["blocks"]))
    tag_show = f"{hc_show['cycle']:02d}{hc_show['hemisphere'][0]}"

    def _mean_density_for(hc, lit_m, train_ds_m, cfg_m, w):
        """Mean-over-K generated density per block: key -> (15,)."""
        keys_hc, cond_hc = block_cond_concat([hc], lit_m, cfg_m, train_ds_m)
        if not keys_hc:
            return {}
        cond_K_hc = cond_hc.repeat_interleave(K, dim=0)
        torch.manual_seed(0)
        samp = sample_empirical_extended(
            lit_m, cond_K_hc, guidance_w=w, bin_width=BIN_WIDTH, device=device,
        ).cpu().numpy().reshape(len(keys_hc), K, 15)
        return {key: samp[i].mean(axis=0) for i, key in enumerate(keys_hc)}

    gen_by_exp = {}
    t_ref = None
    for exp_name in best_by_exp.index:
        row = best_by_exp.loc[exp_name]
        cfg_m = EXPERIMENTS_EMP[exp_name]
        w_m = float(row["guidance_w"])
        lit_m, train_ds_m, _, _ = load_trained_empirical_experiment(
            name=STEM_BY_NAME[exp_name], cfg=cfg_m, windows_aug=windows_aug,
            classical=classical, bin_centers=BIN_CENTERS, ckpt_dir=CKPT_DIR,
            alpha_np=alpha_np, sigma_np=sigma_np, bin_width=BIN_WIDTH,
        )
        mean_dens = _mean_density_for(hc_show, lit_m, train_ds_m, cfg_m, w_m)
        m, t = assemble_butterfly_direct(classical, hc_show, mean_dens, BIN_CENTERS)
        gen_by_exp[exp_name] = (m, t, w_m, float(row["nll_norm_proj_mean"]))
        if t_ref is None or t.size > t_ref.size:
            t_ref = t

    _blk_by_t = {float(b["center_decimal"]): b for b in hc_show["blocks"]}
    emp_show = (np.array([np.histogram(_blk_by_t[t]["lats"], bins=LAT_BINS,
                                       density=True)[0] for t in t_ref])
                if t_ref is not None and t_ref.size else np.empty((0, 15)))

    zero_resid = {(hc_show["cycle"], hc_show["hemisphere"], float(t)): np.zeros(15)
                  for t in (t_ref if t_ref is not None else [])}
    cl_show, _ = assemble_butterfly(classical, hc_show, zero_resid, BIN_CENTERS)

    if t_ref is None or t_ref.size == 0:
        print(f"hemicycle {tag_show}: no windows survive the classical gate.")
    else:
        panels = [("empirical", emp_show, t_ref, None, None),
                  ("classical", cl_show, t_ref, None, nll_cl_val)]
        panels += [(name, m, t, w, nll)
                   for name, (m, t, w, nll) in gen_by_exp.items()]

        vmax = 0.15  # shared color scale across panels — hardcoded for legibility

        n_panels = len(panels)
        ncols = min(3, n_panels)
        nrows = int(np.ceil(n_panels / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows),
                                 sharey=True, squeeze=False)
        axes_flat = axes.flatten()

        for ax, (name, m, t, w, nll) in zip(axes_flat, panels):
            if w is None and nll is None:
                title = "Empirical"
            elif w is None:
                title = "Classical"
            else:
                title = "AI (empirical target)"
            im = ax.imshow(m.T, origin="lower", aspect="auto", vmin=0, vmax=vmax,
                           extent=[t.min(), t.max(),
                                   LAT_BINS[0], LAT_BINS[-1]], cmap="magma")
            ax.axhline(5, color="c", ls=":", lw=1)
            ax.axhline(40, color="c", ls=":", lw=1)
            # ax.set_xlabel("year")
            ax.set_title(title, fontsize=10)
        for ax in axes_flat[len(panels):]:
            ax.axis("off")
        for r in range(nrows):
            axes[r, 0].set_ylabel("|latitude| (°)")

        fig.colorbar(im, ax=axes_flat[:len(panels)].tolist(),
                     fraction=0.025, pad=0.02, label="density")
        fig.suptitle(f"2D Histogram of the Butterfly diagram of Solar Cycle {tag_show}",
                     fontsize=12)
        plt.show()

        print(f"\nPhysical checks for {tag_show}:")
        c = butterfly_physical_checks(emp_show, t_ref, BIN_CENTERS)
        print(f"  {'empirical':>14}: "
              f"sporer_slope={c['sporer_slope']:+.3f} deg/yr "
              f"(equatorward={c['sporer_ok']}), "
              f"in_band={c['in_band_fraction']:.3f}, "
              f"centroid {c['centroid_start']:.1f}->{c['centroid_end']:.1f}")
        c = butterfly_physical_checks(cl_show, t_ref, BIN_CENTERS)
        print(f"  {'classical':>14}: "
              f"sporer_slope={c['sporer_slope']:+.3f} deg/yr "
              f"(equatorward={c['sporer_ok']}), "
              f"in_band={c['in_band_fraction']:.3f}, "
              f"centroid {c['centroid_start']:.1f}->{c['centroid_end']:.1f}, "
              f"NLL={nll_cl_val:.3f}")
        for name, (m, t, w, nll) in gen_by_exp.items():
            c = butterfly_physical_checks(m, t, BIN_CENTERS)
            tag = f"{name} w={w:.1f}"
            print(f"  {tag:>14}: "
                  f"sporer_slope={c['sporer_slope']:+.3f} deg/yr "
                  f"(equatorward={c['sporer_ok']}), "
                  f"in_band={c['in_band_fraction']:.3f}, "
                  f"centroid {c['centroid_start']:.1f}->{c['centroid_end']:.1f}, "
                  f"NLL={nll:.3f}")

    sym_by_exp = {}
    for exp_name in best_by_exp.index:
        row = best_by_exp.loc[exp_name]
        cfg_m = EXPERIMENTS_EMP[exp_name]
        w_m = float(row["guidance_w"])
        lit_m, train_ds_m, _, _ = load_trained_empirical_experiment(
            name=STEM_BY_NAME[exp_name], cfg=cfg_m, windows_aug=windows_aug,
            classical=classical, bin_centers=BIN_CENTERS, ckpt_dir=CKPT_DIR,
            alpha_np=alpha_np, sigma_np=sigma_np, bin_width=BIN_WIDTH,
        )
        checks_by_hc = {}
        for hc in val_hcs:
            gm, gt = assemble_butterfly_direct(
                classical, hc,
                _mean_density_for(hc, lit_m, train_ds_m, cfg_m, w_m),
                BIN_CENTERS,
            )
            if gm.shape[0] >= 2:
                checks_by_hc[(hc["cycle"], hc["hemisphere"])] = \
                    butterfly_physical_checks(gm, gt, BIN_CENTERS)
        sym_by_exp[exp_name] = hemispheric_symmetry(checks_by_hc)

    any_sym = any(sym for sym in sym_by_exp.values())
    if any_sym:
        print("\nHemispheric symmetry (cycles with both N and S in val):")
        for exp_name, sym in sym_by_exp.items():
            w_m = float(best_by_exp.loc[exp_name, "guidance_w"])
            if not sym:
                continue
            print(f"  {exp_name} (w={w_m:.1f}):")
            for cyc, d in sym.items():
                print(f"    cycle {cyc}: "
                      f"|slope_N - slope_S|={d['slope_diff']:.3f}, "
                      f"|in_band_N - in_band_S|={d['in_band_diff']:.3f}")
    else:
        print("\nHemispheric symmetry: no cycle has both N and S in the val "
              "split (val is hemisphere-mixed) — per-hemicycle checks only.")